![Cabec%CC%A7alho_notebook.png](cabecalho_notebook.png)

# PCA - Tarefa 01: *HAR* com PCA

Vamos trabalhar com a base da demonstração feita em aula, mas vamos explorar um pouco melhor como é o desempenho da árvore variando o número de componentes principais.

In [1]:
import pandas as pd

from sklearn.tree import DecisionTreeClassifier

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV


In [ ]:


BASE = "human+activity+recognition+using+smartphones/UCI HAR Dataset/UCI HAR Dataset"

filename_features = f"{BASE}/features.txt"
filename_labels   = f"{BASE}/activity_labels.txt"

filename_subtrain = f"{BASE}/train/subject_train.txt"
filename_xtrain   = f"{BASE}/train/X_train.txt"
filename_ytrain   = f"{BASE}/train/y_train.txt"

filename_subtest  = f"{BASE}/test/subject_test.txt"
filename_xtest    = f"{BASE}/test/X_test.txt"
filename_ytest    = f"{BASE}/test/y_test.txt"


features = pd.read_csv(
    filename_features, sep=r"\s+", header=None, names=['idx', 'nome_var']
)['nome_var']


contagem = {}
nomes_unicos = []
for nome in features:
    if nome in contagem:
        contagem[nome] += 1
        nomes_unicos.append(f"{nome}_{contagem[nome]}")
    else:
        contagem[nome] = 0
        nomes_unicos.append(nome)

print(f"Total de features: {len(nomes_unicos)}")
print(f"Nomes únicos? {len(nomes_unicos) == len(set(nomes_unicos))}")

labels = pd.read_csv(filename_labels, sep=r"\s+", header=None, names=['cod_label', 'label'])

subject_train = pd.read_csv(filename_subtrain, header=None, names=['subject_id']).squeeze("columns")
X_train       = pd.read_csv(filename_xtrain, sep=r"\s+", header=None, names=nomes_unicos)
y_train       = pd.read_csv(filename_ytrain, header=None, names=['cod_label'])

subject_test  = pd.read_csv(filename_subtest, header=None, names=['subject_id']).squeeze("columns")
X_test        = pd.read_csv(filename_xtest, sep=r"\s+", header=None, names=nomes_unicos)
y_test        = pd.read_csv(filename_ytest, header=None, names=['cod_label'])


print(f"\nX_train: {X_train.shape}")  
print(f"X_test : {X_test.shape}")      
print(f"y_train: {y_train.shape}")     
print(f"Atividades: {labels['label'].tolist()}")

Total de features: 561
Nomes únicos? True

X_train: (7352, 561)
X_test : (2947, 561)
y_train: (7352, 1)
Atividades: ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']


## Árvore de decisão

Rode uma árvore de decisão com todas as variáveis, utilizando o ```ccp_alpha=0.001```. Avalie a acurácia nas bases de treinamento e teste. Avalie o tempo de processamento.

In [5]:
%%time


clf = DecisionTreeClassifier(
    ccp_alpha=0.001,    
    random_state=42
)
clf.fit(X_train, y_train['cod_label'])


print(f"Acurácia treino: {clf.score(X_train, y_train['cod_label']):.4f}")
print(f"Acurácia teste : {clf.score(X_test, y_test['cod_label']):.4f}")

Acurácia treino: 0.9758
Acurácia teste : 0.8799
CPU times: total: 6.91 s
Wall time: 7.01 s


## Árvore com PCA

Faça uma análise de componemtes principais das variáveis originais. Utilize apenas uma componente. Faça uma árvore de decisão com esta componente como variável explicativa.

- Avalie a acurácia nas bases de treinamento e teste
- Avalie o tempo de processamento

In [ ]:
%time

prcomp = PCA(n_components=1).fit(X_train)
pc_treino = prcomp.transform(X_train)
pc_teste  = prcomp.transform(X_test)

print(f"Shape pc_treino: {pc_treino.shape}")  
print(f"Shape pc_teste : {pc_teste.shape}")   



CPU times: total: 0 ns
Wall time: 0 ns
Shape pc_treino: (7352, 1)
Shape pc_teste : (2947, 1)


In [9]:

clf = DecisionTreeClassifier(ccp_alpha=0.001, random_state=42)
clf.fit(pc_treino, y_train['cod_label'])


print(f"\nAcurácia treino: {clf.score(pc_treino, y_train['cod_label']):.4f}")
print(f"Acurácia teste : {clf.score(pc_teste, y_test['cod_label']):.4f}")


Acurácia treino: 0.4997
Acurácia teste : 0.4571


## Testando o número de componentes

Com base no código acima, teste a árvore de classificação com pelo menos as seguintes possibilidades de quantidades de componentes: ```[1, 2, 5, 10, 50]```. Avalie para cada uma delas:

- Acurácia nas bases de treino e teste
- Tempo de processamento


In [14]:
import time

componentes = [1, 2, 5, 10, 50]
resultados = []

for n in componentes:
    t0 = time.time()
    
    # PCA
    prcomp = PCA(n_components=n).fit(X_train)
    pc_treino = prcomp.transform(X_train)
    pc_teste  = prcomp.transform(X_test)
    
    # Árvore
    clf = DecisionTreeClassifier(ccp_alpha=0.001, random_state=42)
    clf.fit(pc_treino, y_train['cod_label'])
    
    acc_treino = clf.score(pc_treino, y_train['cod_label'])
    acc_teste  = clf.score(pc_teste,  y_test['cod_label'])
    
    t1 = time.time()
    
    resultados.append({
        'n_components': n,
        'acc_treino': acc_treino,    
        'acc_teste': acc_teste,      
        'tempo_seg': round(t1 - t0, 3)
    })

df_result = pd.DataFrame(resultados)
print(df_result)

   n_components  acc_treino  acc_teste  tempo_seg
0             1    0.499728   0.457075      0.346
1             2    0.612758   0.584662      0.378
2             5    0.846028   0.788599      0.401
3            10    0.892682   0.824228      0.497
4            50    0.919342   0.822871      1.280


## Conclua

- O que aconteceu com a acurácia?
- O que aconteceu com o tempo de processamento?

Sobre a acurácia, ela cresce bem rapido nos primeiros componentes, saindo de 45.7% com 1 componente para 78.9% com 5, e parece que estabiliza a partir de 10 componentes. Adicionar mais componentes além disso não melhora em muita coisa. Olhando o N, entre n=10 e n=50, a acurácia teste fica praticamente igual, mas a árvore começa a sofrer  o que me parece serr overfitting. Mesmo assim, nenhuma versão com PCA superou a árvore completa.
Sobre o tempo para rodar o modelo, ele caiu drasticamente, saindo de de 7 segundos com 561 features iniciais para menos de 1.3s com 50 componentes, e a apenas 0.5s com 10 componentes. O tempo cresce de forma quase linear com o número de componentes.